In [1]:
from ariautils.midi import MidiDict
from ariautils.tokenizer import AbsTokenizer

aria_tokenizer = AbsTokenizer()

midi_file_path = "/mnt/data/improvnet_data/aria-midi-v1-pruned-ext/data/ow/772313_0.mid"
mid = MidiDict.from_midi(midi_file_path)
tokenized_sequence = aria_tokenizer.tokenize(mid)
print(tokenized_sequence[0:100])
print(f"Number of tokens: {len(tokenized_sequence)}")

[('prefix', 'instrument', 'piano'), '<S>', ('piano', 52, 10), ('onset', 0), ('dur', 5000), ('piano', 59, 10), ('onset', 260), ('dur', 5000), ('piano', 64, 20), ('onset', 550), ('dur', 5000), ('piano', 59, 10), ('onset', 820), ('dur', 5000), ('piano', 64, 20), ('onset', 1090), ('dur', 5000), ('piano', 59, 10), ('onset', 1360), ('dur', 4960), ('piano', 52, 20), ('onset', 1650), ('dur', 4670), ('piano', 59, 20), ('onset', 1930), ('dur', 4390), ('piano', 64, 20), ('onset', 2180), ('dur', 4140), ('piano', 59, 10), ('onset', 2450), ('dur', 3870), ('piano', 64, 20), ('onset', 2710), ('dur', 3610), ('piano', 59, 20), ('onset', 2980), ('dur', 3340), ('piano', 40, 20), ('onset', 3290), ('dur', 3030), ('piano', 52, 20), ('onset', 3310), ('dur', 3010), ('piano', 59, 20), ('onset', 3570), ('dur', 2750), ('piano', 64, 20), ('onset', 3830), ('dur', 2490), ('piano', 59, 10), ('onset', 4090), ('dur', 2230), ('piano', 64, 20), ('onset', 4370), ('dur', 1950), ('piano', 59, 20), ('onset', 4620), ('dur', 1

In [ ]:
new_vocab = ()

# Remove all non piano instruments from vocabulary
for token in list(aria_tokenizer.vocab):
    if isinstance(token, tuple) and len(token) == 3:
        if token[0] == "prefix" and token[1] == "instrument" and token[2] != "piano":
            continue
        elif token[0] == "prefix" and token[1] == "composer":
            continue
        elif token[0] == "prefix" and token[1] == "form":
            continue
        elif token[0] == "prefix" and token[1] == "genre":
            continue
        elif token[0] != "piano" and isinstance(token[1], int) and isinstance(token[2], int):
            continue
    new_vocab += (token,)

aria_tokenizer.vocab = new_vocab
vocab_size = len(aria_tokenizer.vocab)
print(f"Vocabulary size: {vocab_size}")

Vocabulary size: 2721


# Aria-dataset

In [1]:
import glob
import os
import json

aria_metadata_filepath = "/data/improvnet/aria-midi-v1-pruned-ext/metadata.json"
with open(aria_metadata_filepath, "r") as f:
    aria_metadata = json.load(f)

# Print a few samples from the config
for key in list(aria_metadata.keys())[:20]:
    print(f"{key}: {aria_metadata[key]}")

# Get all .mid files within multiple directories
aria_midi_files = glob.glob("/data/improvnet/aria-midi-v1-pruned-ext/data/**/*.mid", recursive=True)
print(len(aria_midi_files))

2: {'metadata': {'composer': 'pierné', 'genre': 'classical'}, 'audio_scores': {'0': 0.9721}}
3: {'metadata': {'difficulty': 'advanced'}, 'audio_scores': {'0': 0.9045}}
4: {'metadata': {'composer': 'kondo', 'difficulty': 'advanced', 'music_period': 'modern'}, 'audio_scores': {'0': 0.9757}}
7: {'metadata': {'composer': 'szymanowski', 'opus': 14, 'genre': 'classical', 'form': 'fantasia'}, 'audio_scores': {'0': 0.9647}}
10: {'metadata': {'composer': 'washburn', 'genre': 'classical', 'form': 'waltz', 'difficulty': 'beginner', 'music_period': 'contemporary'}, 'audio_scores': {'0': 0.9394}}
11: {'metadata': {'genre': 'pop'}, 'audio_scores': {'0': 0.9895}}
12: {'metadata': {'composer': 'chabrier', 'genre': 'classical', 'form': 'scherzo', 'piece_number': 10}, 'audio_scores': {'0': 0.719}}
13: {'metadata': {'composer': 'sheeran', 'genre': 'pop', 'music_period': 'modern'}, 'audio_scores': {'0': 0.9875}}
15: {'metadata': {'genre': 'pop', 'music_period': 'modern'}, 'audio_scores': {'0': 0.8136}}
16

In [2]:
aria_data = []
for midi_file in aria_midi_files:
    filename = os.path.basename(midi_file).split("_")[0]
    if filename in aria_metadata:
        entry = {
            "midi_filepath": midi_file,
            "genre": aria_metadata[filename].get('metadata', {}).get('genre', None).lower() if aria_metadata[filename].get('metadata', {}).get('genre', None) else None,
            "composer": aria_metadata[filename].get('metadata', {}).get('composer', None).lower() if aria_metadata[filename].get('metadata', {}).get('composer', None) else None,
            "form": aria_metadata[filename].get('metadata', {}).get('form', None).lower() if aria_metadata[filename].get('metadata', {}).get('form', None) else None,
            "musical_period": aria_metadata[filename].get('metadata', {}).get('musical_period', None).lower() if aria_metadata[filename].get('metadata', {}).get('musical_period', None) else None,
        }
        aria_data.append(entry)

In [3]:
aria_data[0:2]  # Display first two entries

[{'midi_filepath': '/data/improvnet/aria-midi-v1-pruned-ext/data/qf/843627_0.mid',
  'genre': 'pop',
  'composer': None,
  'form': None,
  'musical_period': None},
 {'midi_filepath': '/data/improvnet/aria-midi-v1-pruned-ext/data/qf/842785_0.mid',
  'genre': 'classical',
  'composer': 'liszt',
  'form': 'fantasia',
  'musical_period': None}]

In [4]:
import csv

with open("/data/improvnet/maestro-v3.0.0/maestro-v3.0.0.csv", "r", newline='') as csvfile:
    reader = csv.DictReader(csvfile)
    maestro_metadata = {row['midi_filename']: row for row in reader}    

In [5]:
maestro_data = []
for key, value in maestro_metadata.items():
    midi_filepath = os.path.join("/data/improvnet/maestro-v3.0.0/", value['midi_filename'])
    forms = ['sonata', 'etude', 'waltz', 'nocturne', 'prelude', 'fugue', 'suite', 'ballade', 'mazurka', 'polonaise', 'scherzo', 'fantasy', 'fughetta', 'impromptu', 'variation']
    form = None
    for f in forms:
        if f.lower() in value.get('canonical_title', '').lower():
            form = f.lower()
            break
    entry = {
        "midi_filepath": midi_filepath,
        "genre": "classical",
        "composer": value.get('canonical_composer', None).lower(),
        "form": form,
        "musical_period": None,
    }
    maestro_data.append(entry)

In [6]:
# Get all .mid files within multiple directories
pijama_midi_files = glob.glob("/data/improvnet/pijama-retranscribed/data/**/*.mid", recursive=True)
print(len(pijama_midi_files))

2736


In [7]:
pijama_data = []
for midi_file in pijama_midi_files:
    filename = os.path.basename(midi_file)
    entry = {
        "midi_filepath": midi_file,
        "genre": "jazz",
        "composer": None,
        "form": None,
        "musical_period": None,
    }
    pijama_data.append(entry)

In [8]:
# Get all .mid files within multiple directories
doug_midi_files = glob.glob("/data/improvnet/doug_mcenzie_jazz/**/*.mid", recursive=True)
print(len(doug_midi_files))

# Remove files -> "My Old FlameGM.mid", "Whilewereyoung.mid"
doug_midi_files = [f for f in doug_midi_files if "My Old FlameGM.mid" not in f and "Whilewereyoung.mid" not in f]

299


In [9]:
doug_data = []
for midi_file in doug_midi_files:
    filename = os.path.basename(midi_file)
    entry = {
        "midi_filepath": midi_file,
        "genre": "jazz",
        "composer": None,
        "form": None,
        "musical_period": None,
    }
    doug_data.append(entry)

In [10]:
# Get all .mid files within multiple directories
symphony_files = glob.glob("/data/improvnet/SymphonyNet_Dataset/**/*.mid", recursive=True)
print(len(symphony_files))

46360


In [11]:
symphony_data = []
for midi_file in symphony_files:
    filename = os.path.basename(midi_file)
    if "classical" in midi_file:
        entry = {
            "midi_filepath": midi_file,
            "genre": "classical",
            "composer": None,
            "form": None,
            "musical_period": None,
        }
    else:
        entry = {
            "midi_filepath": midi_file,
            "genre": None,
            "composer": None,
            "form": None,
            "musical_period": None,
        }
    symphony_data.append(entry)

In [12]:
# Get all .mid files within multiple directories
classicalarchive_files = glob.glob("/data/improvnet/ClassicalArchives-MIDI-Collection/**/*.mid", recursive=True)
print(len(classicalarchive_files))

8135


In [13]:
classicalarchive_data = []
for midi_file in classicalarchive_files:
    filename = os.path.basename(midi_file)
    entry = {
        "midi_filepath": midi_file,
        "genre": "classical",
        "composer": None,
        "form": None,
        "musical_period": None,
    }
    classicalarchive_data.append(entry)

In [14]:
all_data = aria_data + maestro_data + pijama_data + doug_data + symphony_data + classicalarchive_data
print(f"Total number of MIDI files collected: {len(all_data)} from {len(aria_data)} (Aria) + {len(maestro_data)} (Maestro) + {len(pijama_data)} (Pijama) + {len(doug_data)} (Doug McKenzie) + {len(symphony_data)} (SymphonyNet) + {len(classicalarchive_data)} (ClassicalArchive)")

Total number of MIDI files collected: 791493 from 732689 (Aria) + 1276 (Maestro) + 2736 (Pijama) + 297 (Doug McKenzie) + 46360 (SymphonyNet) + 8135 (ClassicalArchive)


In [15]:
# Get count of all unique genre, composers and form and print them

unique_genres = set()
unique_composers = set()
unique_forms = set()
for entry in all_data:
    if entry['genre']:
        unique_genres.add(entry['genre'])
    if entry['composer']:
        unique_composers.add(entry['composer'])
    if entry['form']:
        unique_forms.add(entry['form'])

print(f"Unique genres: {unique_genres}")
print(f"Unique composers: {unique_composers}")
print(f"Unique forms: {unique_forms}")

Unique genres: {'ragtime', 'ambient', 'jazz', 'folk', 'atonal', 'blues', 'rock', 'pop', 'soundtrack', 'classical'}
Unique composers: {'falcés', 'samuels', 'yakazera', 'hafner', 'bartók', 'гнесина', 'erttaş', 'champagne', 'cyrin', 'thelivingtombstone', 'desyatnikov', 'domínguez', 'mccree', 'farkas', 'koda', 'scholfield', 'liadow', 'goroyan', 'kiryuuin', 'mirzoyan', 'van_heussen', 'kaho', 'cuppix', 'salazar', 'clergue', 'lecoz', 'becho', 'broderick', 'мартынов', 'giga-p', 'kumi', 'kravchuk', 'stump', 'maretov', 'sparks', 'krebs', 'omelchuk', 'ara', 'maoukon', 'irie', 'fujikura', 'morleo', 'kwangmin', 'perabo', 'macgregor', 'sawer', 'emery', 'cerati', 'hirota', 'manna', 'zidani', 'alers', 'bogoyavlensky', 'dicolia', 'coates', 'siskind', 'crabtree', 'monorosi', 'seoyizi', 'mitsoshi', 'daly', 'targosza', 'barcelata', 'agostino', 'line', 'nedbal', 'tanikawa', 'mitsuji', 'yein', 'jacques-dalcroze', 'buka', 'kullhau', 'taussat', 'jim_chappell', 'takagi', 'leclair', 'mielck', 'appermont', 'kaya

In [17]:
import random

# Write all_data to a JSONL file
with open("/home/ubuntu/keshav/improvnet_2/improvnet/data/misc_data.jsonl", "w") as f:
    for entry in all_data:
        # Merge forms
        if entry['form'] == "fantasia":
            entry['form'] = "fantasy"
        
        # Train (95%), validation (2%), test split (3%)
        rand_val = random.random()
        if rand_val < 0.97:
            entry['split'] = 'train'
        elif rand_val < 0.99:
            entry['split'] = 'validation'
        else:
            entry['split'] = 'test'

        f.write(json.dumps(entry) + "\n")

## GigaMIDI

In [18]:
import sys
import csv
import os

csv.field_size_limit(sys.maxsize)

csv_filepath = "/data/improvnet/GigaMIDI/Final_GigaMIDI_V1.1_Final/Final-Metadata-Extended-GigaMIDI-Dataset-updated.csv"

# Read CSV file and print file_path and music_styles_curated keys
with open(csv_filepath, "r") as csvfile:
    reader = csv.DictReader(csvfile)
    # Create data entries for each row
    gigamidi_data = []
    for row in reader:
        # midi_filepath = row['file_path']
        # './Final_GigaMIDI_V1.1_Final/training-V1.1-80%/no-drums/4/81a8984f7cac6fac511e917fe6d307de.mid'
        midi_filepath = os.path.join("/data/improvnet/GigaMIDI/Final_GigaMIDI_V1.1_Final/", row['file_path'].lstrip("./Final_GigaMIDI_V1.1_Final/"))
        genres = row['music_styles_curated'].lower().split(";") if row['music_styles_curated'] else []
        genre = genres[0] if genres else None
        entry = {
            "midi_filepath": midi_filepath,
            "genre": genre,
            "composer": row['artist'].lower() if row['artist'] else None,
            "form": None,
            "musical_period": None,
        }
        gigamidi_data.append(entry)

print(f"Total number of GigaMIDI MIDI files collected: {len(gigamidi_data)}")

Total number of GigaMIDI MIDI files collected: 2136218


In [19]:
# Count unique genres in gigamidi_data
genre_count = {}
for entry in gigamidi_data:
    genre = entry['genre']
    if genre:
        if genre in genre_count:
            genre_count[genre] += 1
        else:
            genre_count[genre] = 1

# Print genre counts
for genre, count in genre_count.items():
    print(f"{genre}: {count}")

classical: 43916
metal: 1032
game: 21697
world: 297
downtempo: 55
rock: 1956
punk: 642
rap: 146
dance: 92
blues: 66
house: 96
pop: 481
breakbeat: 18
edm: 4
latin: 102
country: 244
alternative: 6
reggae: 86
jazz: 191
trance: 63
disco: 295
techno: 94
soundtrack: 86
folk: 102
drum&bass: 24


In [20]:
gigamidi_data[0:2]  # Display first two entries

[{'midi_filepath': '/data/improvnet/GigaMIDI/Final_GigaMIDI_V1.1_Final/training-V1.1-80%/no-drums/4/81a8984f7cac6fac511e917fe6d307de.mid',
  'genre': None,
  'composer': None,
  'form': None,
  'musical_period': None},
 {'midi_filepath': '/data/improvnet/GigaMIDI/Final_GigaMIDI_V1.1_Final/training-V1.1-80%/no-drums/4/d0e3796eb5c2bb37da4fb12433b29949.mid',
  'genre': None,
  'composer': None,
  'form': None,
  'musical_period': None}]

In [21]:
import json
import random

genres = {'rock', 'classical', 'ragtime', 'pop', 'blues', 'soundtrack', 'folk', 'atonal', 'ambient', 'jazz', 'metal', 'game'}

# Write gigamidi_data to a JSONL file and for genres other than the above, set genre to None
with open("/home/ubuntu/keshav/improvnet_2/improvnet/data/gigamidi_data.jsonl", "w") as f:
    for entry in gigamidi_data:
        if entry['genre'] not in genres:
            entry['genre'] = None
        
        # Train (95%), validation (2%), test split (3%)
        rand_val = random.random()
        if rand_val < 0.97:
            entry['split'] = 'train'
        elif rand_val < 0.99:
            entry['split'] = 'validation'
        else:
            entry['split'] = 'test'

        f.write(json.dumps(entry) + "\n")